# LLM judge로 궤적 평가

이 노트북은 Python 3.12 새 커널에서 독립 실행합니다.

## 패키지 설치

In [1]:
# 노트북이 사용하는 패키지 버전을 새 커널 환경에 설치합니다.
%pip install -q agentevals==0.0.9 langchain==1.4.0 langchain-openai==1.6.0 langgraph==1.2.11 python-dotenv==1.2.3

Note: you may need to restart the kernel to use updated packages.


## 환경 준비

In [2]:
# 공개하지 않은 API 키는 존재 여부만 확인하고 고정한 패키지 버전을 출력합니다.
import os
from pathlib import Path
from importlib.metadata import version
from dotenv import load_dotenv
for folder in [Path.cwd(), *Path.cwd().parents]:
    if (folder / ".env").is_file():
        load_dotenv(folder / ".env")
        break
print("agentevals", version("agentevals"))
print("langchain", version("langchain"))
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY가 필요합니다."

agentevals 0.0.9
langchain 1.4.0


## 평가할 궤적

In [3]:
# judge와 결정적 평가가 같은 실제·기준 메시지를 읽도록 입력을 고정합니다.
from langchain.messages import HumanMessage, AIMessage, ToolMessage
def trace(*tools: str) -> list:
    messages=[HumanMessage("규정을 확인해 주세요.")]
    for i,name in enumerate(tools,1):
        messages += [AIMessage("", tool_calls=[{"name":name,"args":{"topic":"대출"},"id":f"c{i}","type":"tool_call"}]), ToolMessage("확인", tool_call_id=f"c{i}")]
    messages.append(AIMessage("일반 도서는 14일입니다."))
    return messages
reference=trace("search_rules", "check_source")
actual=trace("search_rules","check_source")

## 실제 LLM judge 실행

In [4]:
# TRAJECTORY_ACCURACY_PROMPT로 전체 단계의 논리와 효율을 실제 모델이 판정합니다.
from agentevals.trajectory.llm import create_trajectory_llm_as_judge, TRAJECTORY_ACCURACY_PROMPT
evaluator=create_trajectory_llm_as_judge(model="openai:gpt-5.4-mini",prompt=TRAJECTORY_ACCURACY_PROMPT)
evaluation=evaluator(outputs=actual,reference_outputs=reference)
print(evaluation)
assert evaluation["score"] in [True, False]

{'key': 'trajectory_accuracy', 'score': False, 'comment': "The user asked to check the regulations, but the trajectory shows the assistant arbitrarily deciding the topic is '대출' (loan) without evidence from the user message. It then performs two tool calls that only return generic confirmation and finally answers with a specific rule, '일반 도서는 14일입니다' (general books are 14 days), which is unrelated to the inferred loan topic and unsupported by the tool outputs. The steps do not form a logical progression toward the user's request, and the final answer does not match the likely goal. Thus, the score should be: false.", 'metadata': None}


## 결정적 검사와 함께 보기

In [5]:
# 같은 궤적의 strict 점수와 judge 설명을 나란히 출력해 구조와 의미 평가를 구분합니다.
from agentevals.trajectory.match import create_trajectory_match_evaluator
deterministic=create_trajectory_match_evaluator(trajectory_match_mode="strict")(outputs=actual,reference_outputs=reference)
print("strict:", deterministic)
print("judge:", evaluation)
assert deterministic["score"] is True

strict: {'key': 'trajectory_strict_match', 'score': True, 'comment': None, 'metadata': None}
judge: {'key': 'trajectory_accuracy', 'score': False, 'comment': "The user asked to check the regulations, but the trajectory shows the assistant arbitrarily deciding the topic is '대출' (loan) without evidence from the user message. It then performs two tool calls that only return generic confirmation and finally answers with a specific rule, '일반 도서는 14일입니다' (general books are 14 days), which is unrelated to the inferred loan topic and unsupported by the tool outputs. The steps do not form a logical progression toward the user's request, and the final answer does not match the likely goal. Thus, the score should be: false.", 'metadata': None}
